# Inspect Overture divisions GeoParquet files

This notebook scans the `theme=divisions` release folder for GeoParquet files, reports their schema, and shows a handful of sample rows using DuckDB. Each subdirectory is sampled once because the files within share the same structure.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for parent in (start, *start.parents):
        if (parent / '.git').exists():
            return parent
    raise RuntimeError(f'Could not find repository root from {start}')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_DIR = REPO_ROOT / 'data'
RESULTS_DIR = DATA_DIR / 'results'
GIS_DIR = REPO_ROOT / 'gis_data'
MODULE_ROOT = REPO_ROOT / 'code' / 'overture_analysis'

if str(MODULE_ROOT) not in sys.path:
    sys.path.append(str(MODULE_ROOT))

RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from collections import defaultdict

import duckdb
from IPython.display import display

DIVISIONS_RELEASE = '2025-08-20.1'
BASE_PATH = GIS_DIR / 'overturemaps-us-west-2' / 'release' / DIVISIONS_RELEASE / 'theme=divisions'
print(f'Base directory: {BASE_PATH}')

parquet_files = sorted(BASE_PATH.rglob('*.parquet'))
if not parquet_files:
    raise FileNotFoundError('No GeoParquet files found under the divisions theme directory.')

files_by_dir = defaultdict(list)
for file_path in parquet_files:
    files_by_dir[file_path.parent].append(file_path)

print(f'Found {len(parquet_files)} parquet files across {len(files_by_dir)} directories.')
for directory, files in sorted(files_by_dir.items()):
    rel_dir = directory.relative_to(BASE_PATH)
    print(f'{rel_dir}: {len(files)} file(s)')


In [ ]:
con = duckdb.connect(database=':memory:')

for directory, files in sorted(files_by_dir.items()):
    sample_file = files[0]
    rel_dir = directory.relative_to(BASE_PATH)
    print(f"
=== {rel_dir} ===")
    print(f"Sample file: {sample_file.name}")
    schema_df = con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)", [str(sample_file)]
    ).fetchdf()
    display(schema_df)
    sample_rows_df = con.execute(
        "SELECT * FROM read_parquet(?) LIMIT 5", [str(sample_file)]
    ).fetchdf()
    display(sample_rows_df)


In [ ]:
division_path = BASE_PATH / 'type=division'
country_pattern = str(division_path / '*.parquet')
countries_output = RESULTS_DIR / 'countries.parquet'

con = duckdb.connect(database=':memory:')
country_df = con.execute(
    "SELECT * FROM read_parquet(?) WHERE subtype = 'country' ORDER BY id",
    [country_pattern]
).fetchdf()
print(f'Retrieved {len(country_df)} division records with subtype=country.')
country_df.to_parquet(countries_output, index=False)
print(f'Saved results to {countries_output}')
display(country_df.head())
con.close()


In [ ]:
import geopandas as gpd
import folium
from folium.features import GeoJsonTooltip
from IPython.display import display

countries_gdf = gpd.read_parquet(countries_output)
if countries_gdf.crs is None or countries_gdf.crs.to_epsg() != 4326:
    countries_gdf = countries_gdf.to_crs('EPSG:4326')

m = folium.Map(location=[0, 0], zoom_start=2)
folium.GeoJson(
    countries_gdf,
    tooltip=GeoJsonTooltip(fields=['country'], aliases=['Country:']),
).add_to(m)
display(m)
